# Optimal Power flow

#### Parameter Definitions

In our data file, we have lots of values, and multiple sheets. Take a moment to make sure you understand what every parameter means, conceptually. Below you can find the relevant parameter definitions.

##### Node (i.e. Bus) Data
**Parameter Name** | **Description**
---|---
Vmax        | Maximal voltage of node
Vmin        | Minimal voltage of node
VAnglemax   | Maximal voltage angle of node
VAnglemin   | Minimal voltage angle of node
Pload       | Real power drawn by loads at node
Qload       | Reactive power drawn by loads at node
PGmax       | Maximal real power of generators at node
PGmin       | Minimal real power of generators at node
QGmax       | Maximal reactive power of generators at node
QGmin       | Minimal reactive power of generators at node
a           | Generator cost function first coefficient# a*p^2
b           | Generator cost function second coefficient# b*p
c           | Generator cost function third coefficient# c

where the cost function of a dispatchable generator at node $i$, with a generation power of $p_i$, is defined as

$$ C_i(p_i) = a_i p_i^2 + b_i p_i + c_i $$

subject to the power constraints $ P^{G,min}_i \le p_i \le P^{Gmax}_i $.

**Note**: The slack node, i.e. the reference node, serves as an angular reference for all other nodes in the system, and is thus set to 0 deg. At this slack node, the voltage magnitude is also assumed to be 1.0 p.u.

##### Admittance Matrices

To model the admittance between two buses, i and k, we have set up the following matrices.
One matrix is for the real part of the admittance, while the other is for the imaginary part.
Since these matrices cover all relationships between buses, these are square matrices.
An example for the structure of the Real Admittance Matrix is shown below:

$$\begin{bmatrix}
G_{1,1} & G_{1,2} & \dots & \dots & G_{1,n} \\
G_{2,1} & \ddots & & & \vdots \\
\vdots & & G_{i,k} & & \vdots \\
\vdots & & & \ddots & G_{n-1,n} \\
G_{n,1} & \dots & \dots & G_{n,n-1} & G_{n,n} 
\end{bmatrix}$$

**Knowledge Checkup:**
To make sure you understand the data structure, take a look at the data file(s) and see if you can answer the following conceptual questions:

1) How many buses are there in total? -> 5
2) Which buses have generators connected to them? -> 1,3,4
3) Which buses have loads connected to them? -> 4,5
4) Which one of these nodes is the slack node? -> 1

After you have answered these questions for yourself, you're all ready to code!

#### Dependency Setup

In [93]:
import os
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import cos, sin
from opf_utils import solve_and_print, create_results_json

# Your email address (required for NEOS)
os.environ['NEOS_EMAIL'] = 'jonas.streuff@tum.de'
FILENAME = 'data//OPFData.xlsx'

### Test

In [94]:
test = pd.read_excel(FILENAME, sheet_name='NodeData')
test.head()

,Vmax,Vmin,VAnglemax,VAnglemin,Pload,Qload,PGmax,PGmin,QGmax,QGmin,a,b,c
0,1.00,1.00,0.000000,0.000000,0.000,0.000,1000.0,-1000.00,1000.0,-1000.0,0.0,0.35,0
1,1.05,0.95,3.141593,-3.141593,0.000,0.000,0.0,0.00,0.0,0.0,0.0,0.00,0
2,1.05,0.95,3.141593,-3.141593,0.000,0.000,0.4,0.10,0.3,-0.2,0.4,0.20,0
3,1.05,0.95,3.141593,-3.141593,0.900,0.400,0.4,0.05,0.2,-0.2,0.5,0.30,0
4,1.05,0.95,3.141593,-3.141593,0.239,0.129,0.0,0.00,0.0,0.0,0.0,0.00,0


#### Model Setup

In [95]:
def add_node_data(filename, m):
    # TODO: Load the pandas dataframes for the node data.
    # Hint: Pay attention to the file types and headers. 
    data = pd.read_excel(filename, sheet_name='NodeData')

    # TODO: Set the number of nodes
    m.nodes = pyo.Set(initialize=data.index.tolist())
    
    # TODO: Complete this function
    m.V_max = pyo.Param(m.nodes, initialize=data['Vmax'].to_dict())
    m.V_min = pyo.Param(m.nodes, initialize=data['Vmin'].to_dict())

    m.V_angle_max = pyo.Param(m.nodes, initialize=data['VAnglemax'].to_dict())
    m.V_angle_min = pyo.Param(m.nodes, initialize=data['VAnglemin'].to_dict())
    m.P_load = pyo.Param(m.nodes, initialize=data['Pload'].to_dict())
    m.Q_load = pyo.Param(m.nodes, initialize=data['Qload'].to_dict())

    m.P_G_max = pyo.Param(m.nodes, initialize=data['PGmax'].to_dict())
    m.P_G_min = pyo.Param(m.nodes, initialize=data['PGmin'].to_dict())
    
    m.Q_G_max = pyo.Param(m.nodes, initialize=data['QGmax'].to_dict())
    m.Q_G_min = pyo.Param(m.nodes, initialize=data['QGmin'].to_dict())

    m.a = pyo.Param(m.nodes, initialize=data['a'].to_dict())
    m.b = pyo.Param(m.nodes, initialize=data['b'].to_dict())
    m.c = pyo.Param(m.nodes, initialize=data['c'].to_dict())

    return m

def add_real_admittance_data(filename, m):
    # TODO: Load the pandas dataframes for the node data.
    # Hint: Pay attention to the file types and headers.
    data = pd.read_excel(filename, sheet_name='RealAdmittanceMatrix', header=None)
    assert data.shape[0] == data.shape[1], "The admittance data is not a square matrix"

    # TODO: Complete this function
    # We must flatten the dataframe to a dictionary for Pyomo initialization
    G_dict = {}
    rows, cols = data.shape
    for r in range(rows):
        for c in range(cols):
            G_dict[r, c] = data.iat[r, c]
    
    # Initialize the Parameter G indexed by (node, node)
    m.G = pyo.Param(m.nodes, m.nodes, initialize=G_dict)
    return m

def add_imag_admittance_data(filename, m):
    # TODO: Load the pandas dataframes for the node data.
    # Hint: Pay attention to the file types and headers.
    data = pd.read_excel(filename, sheet_name='ImaginaryAdmittanceMatrix', header=None)
    assert data.shape[0] == data.shape[1], "The admittance data is not a square matrix"

    # TODO: Complete this function
    # We must flatten the dataframe to a dictionary for Pyomo initialization
    
    B_dict = {}
    rows, cols = data.shape
    for r in range(rows):
        for c in range(cols):
            B_dict[r, c] = data.iat[r, c]

    m.B = pyo.Param(m.nodes, m.nodes, initialize=B_dict)

    return m

#### AC Optimal Power Flow Setup

In [96]:
def objective_rule(m):
    # TODO: Complete this function
    # Minimize sum of costs C_i(P_Gi) = a*P_Gi^2 + b*P_Gi + c
    return sum(
        m.a[i] * m.x_p[i]**2 + m.b[i] * m.x_p[i] + m.c[i]
        for i in m.nodes
    )

# TODO: Add functions for the AC OPF constraint rules, if necessary.
# Helper rules for linear constraints (inequalities)
def power_bounds_rule(m, i):
    # P_min <= P_G <= P_max
    return (m.P_G_min[i], m.x_p[i], m.P_G_max[i])

def reactive_power_bounds_rule(m, i):
    # Q_min <= Q_G <= Q_max
    return (m.Q_G_min[i], m.x_q[i], m.Q_G_max[i])

def voltage_bounds_rule(m, i):
    # V_min <= V <= V_max
    return (m.V_min[i], m.x_v[i], m.V_max[i])

def voltage_angle_bounds_rule(m, i):
    # delta_min <= delta <= delta_max
    return (m.V_angle_min[i], m.x_v_angle[i], m.V_angle_max[i])

def add_linear_constraints_ac(m):
    # TODO: Complete this function

    # Generator Real Power Limits
    m.power_bounds = pyo.Constraint(m.nodes, rule=power_bounds_rule)

    # Voltage Phase Angle Limits
    m.voltage_angle_bounds = pyo.Constraint(m.nodes, rule=voltage_angle_bounds_rule)

    # Generator Reactive Power Limits
    m.reactive_power_bounds = pyo.Constraint(m.nodes, rule=reactive_power_bounds_rule)

    # Bus Voltage Magnitude Limits
    m.voltage_bounds = pyo.Constraint(m.nodes, rule=voltage_bounds_rule)

    return m

# Helper rules for non-linear constraints (Power Balance)
def active_power_balance_rule(m, i):
    # P_i(V, delta) equation
    # P_flow_i = V_i * sum( V_k * (G_ik * cos(d_i - d_k) + B_ik * sin(d_i - d_k)))
    P_flow_i = m.x_v[i] * sum(
        m.x_v[k] * (
            m.G[i, k] * cos(m.x_v_angle[i] - m.x_v_angle[k]) + 
            m.B[i, k] * sin(m.x_v_angle[i] - m.x_v_angle[k])
        )
        for k in m.nodes
    )

    # p_G - P_L - P_flow = 0
    return m.x_p[i] - m.P_load[i] - P_flow_i == 0

def reactive_power_balance_rule(m, i):
    # Q_i(V, delta) equation
    # Q_flow_i = V_i * sum( V_k * (G_ik * sin(d_i - d_k) - B_ik * cos(d_i - d_k)) )
    Q_flow_i = m.x_v[i] * sum(
        m.x_v[k] * (
            m.G[i, k] * sin(m.x_v_angle[i] - m.x_v_angle[k]) - 
            m.B[i, k] * cos(m.x_v_angle[i] - m.x_v_angle[k])
        )
        for k in m.nodes
    )
    
    # Q_G - Q_L - Q_flow = 0
    return m.x_q[i] - m.Q_load[i] - Q_flow_i == 0

def add_nonlinear_constraints_ac(m):
    # TODO: Complete this function

    # Active Power Balance
    m.power_balance = pyo.Constraint(m.nodes, rule=active_power_balance_rule)

    # Reactive Power Balance
    m.reactive_power_balance = pyo.Constraint(m.nodes, rule=reactive_power_balance_rule)

    return m

##### DC Optimal Power Flow Setup

In [97]:
# TODO: Add functions for the DC OPF constraint rules, if necessary.

# Helper rules for DC Linear Constraints (Bounds)
def power_bounds_rule_dc(m, i):
    # P_min <= P_G <= P_max
    # Same as AC, but specific to the DC model setup
    return (m.P_G_min[i], m.x_p[i], m.P_G_max[i])

def voltage_angle_bounds_rule_dc(m, i):
    # delta_min <= delta <= delta_max
    # Same as AC but specific to the DC model setup
    return (m.V_angle_min[i], m.x_v_angle[i], m.V_angle_max[i])

# Helper rule for DC Power Balance 
def active_power_balance_rule_dc(m, i):
    # P_i(delta) = sum( B_ik * (delta_i - delta_k) )
    # Assumptions V=1.0, G=0, small angles 
    P_flow_i = sum(
        m.B[i, k] * (m.x_v_angle[i] - m.x_v_angle[k])
        for k in m.nodes
    )

    # P_G - P_L - P_flow = 0 
    return m.x_p[i] - m.P_load[i] - P_flow_i == 0

def add_linear_constraints_dc(m):
    # TODO: Complete this function, similar to add_linear_constraints_ac
    
    # Generator Active Power Limits
    m.power_bounds = pyo.Constraint(m.nodes, rule=power_bounds_rule_dc)
    
    # Voltage Angle Limits
    m.voltage_angle_bounds = pyo.Constraint(m.nodes, rule=voltage_angle_bounds_rule_dc)
    
    # Node: Voltage Magnitude and Reactive Power constraints are removed
    return m

def add_nonlinear_constraints_dc(m):
    # TODO: Complete this function, similar to add_nonlinear_constraints_ac
    # Note: The DC power flow equations are actually linear

    # Active Power Balance
    m.power_balance = pyo.Constraint(m.nodes, rule=active_power_balance_rule_dc)

    return m

##### Load Model from Data

In [98]:
# Loads the data from the given file into the model, defines the optimization problem, and returns the model at the end.
def load_model(filename, m, AC=True):
    # 1. Fill the model with the parameters from the files
    # Loading data from the excel sheets
    m = add_node_data(filename, m)
    m = add_real_admittance_data(filename, m)
    m = add_imag_admittance_data(filename, m)

    # 2. Define optimization variables
    
    # Active Power Generation 
    m.x_p = pyo.Var(m.nodes,
                    within=pyo.Reals,
                    doc='Optimization variable (power generation)')

    # Reactive Power Generation
    m.x_q = pyo.Var(m.nodes,
                    within=pyo.Reals,
                    doc='Optimization variable (reactive power generation)')
    
    # Voltage Magnitude (V)
    m.x_v = pyo.Var(m.nodes,
                    within=pyo.Reals,
                    doc='Optimization variable (voltage)')

    # Voltage Angle (delta)
    m.x_v_angle = pyo.Var(m.nodes,
                         within=pyo.Reals,
                         doc='Optimization variable (voltage angle)')
    
    # 3. Define objective function
    # Minimize sum of generator costs
    m.objective_function = pyo.Objective(
        rule=objective_rule,
        sense=pyo.minimize,
        doc='minimize(cost = sum of all generator operating costs)')
    
    # 4. Define the linear constraints
    if AC:
        m = add_linear_constraints_ac(m)
    else:
        # Added manually 
        for i in m.nodes:
            m.x_v[i].fix(1.0) # Voltage magnitude assumed 1.0 p.u.

        m = add_linear_constraints_dc(m)
    
    # Define the nonlinear constraints
    if AC:
        m = add_nonlinear_constraints_ac(m)
    else:
        # Added manually 
        for i in m.nodes:
            m.x_v[i].fix(1.0) # Voltage magnitude assumed 1.0 p.u.
            
        m = add_nonlinear_constraints_dc(m)

    return m

##### Solve AC Model

In [99]:
AC = True
model = load_model(FILENAME, pyo.ConcreteModel(), AC)
solve_and_print(model, AC)
# TODO: Uncomment if you want to write the result to json file
team_name = 'Group_B' 
create_results_json(model, team_name, AC)

model.name="unknown";
    - termination condition: optimal
    - message from solver: CONOPT 4.36\x3a Locally optimal; objective
      0.4032767007444491; 7 iterations; evals\x3a nf = 8, ng = 7, nc = 50, nJ
      = 13, nH = 1, nHv = 10



Solution


Node		Voltage angle [deg]
1 		 0.0
2 		 -0.1479605917142426
3 		 -0.06511621773008298
4 		 -0.1622079660764928
5 		 -0.15284194533566098

Node		Voltage magnitude
1 		 1.0
2 		 0.9839069187431303
3 		 0.9652306188492912
4 		 0.9710647027190298
5 		 0.95

Node		Power
1 		 0.9451009564369689
2 		 0.0
3 		 0.19409263572233498
4 		 0.05666246887045961
5 		 0.0

Node		Reactive Power
1 		 0.6099494016115005
2 		 0.0
3 		 -0.08980595382293306
4 		 0.2
5 		 0.0

Cost:  0.4032767007444491
Results written to Group_B_AC.json


##### Solve DC Model

In [100]:
AC = False
model = load_model(FILENAME, pyo.ConcreteModel(), AC)
solve_and_print(model, AC)
# TODO: Uncomment if you want to write the result to json file
team_name = 'Group_B' 
create_results_json(model, team_name, AC)




Solution


Node		Voltage angle [deg]
1 		 0.0
2 		 -0.1414593802110106
3 		 -0.06395843445725626
4 		 -0.1570625513306749
5 		 -0.1458868031946045

Node		Power
1 		 0.9015
2 		 0.0
3 		 0.1874999999999999
4 		 0.05
5 		 0.0

Cost:  0.3833374999999999
Results written to Group_B_DC.json
